In [3]:
from google_play_scraper import reviews, app, Sort
import pandas as pd, time, os

APPS = {
    "Vodafone Egypt": {"id": "com.emeint.android.myservices",          "domain": "Telecom"},
    "Orange Egypt":   {"id": "com.orange.mobinilandme",                "domain": "Telecom"},
    "WE":             {"id": "com.ucare.we",                           "domain": "Telecom"},
    "Etisalat Cash":  {"id": "com.etisalat.flous",                     "domain": "Payments"},
    "Instapay":       {"id": "com.egyptianbanks.instapay",             "domain": "Payments"},
    "Fawry":          {"id": "com.fawry.myfawry",                      "domain": "Payments"},
    "NBE":            {"id": "com.ofss.obdx.and.nbe.com.eg",           "domain": "Banking"},
    "CIB":            {"id": "com.cibeg.ddc1.digitalbanking.live",     "domain": "Banking"},
    "Banque Misr":    {"id": "com.BanqueMisr.MobileBanking",           "domain": "Banking"},
    "Amazon":         {"id": "com.amazon.mShop.android.shopping",      "domain": "E-commerce"},
    "Jumia":          {"id": "com.jumia.android",                      "domain": "E-commerce"},
    "Noon":           {"id": "com.noon.buyerapp",                      "domain": "E-commerce"},
    "Uber":           {"id": "com.ubercab",                            "domain": "Transport Services"},
    "inDrive":        {"id": "sinet.startup.inDriver",                 "domain": "Transport Services"},
    "Careem":         {"id": "com.careem.acma",                        "domain": "Transport Services"},
    "DiDi":           {"id": "com.didiglobal.passenger",               "domain": "Transport Services"},
}

def review_type(r):
    return "Negative" if r <= 2 else "Neutral" if r == 3 else "Positive"

def fetch_app(app_name, info, target=4000):
    rows, token = [], None
    while len(rows) < target:
        batch, token = reviews(info["id"], lang="ar", country="eg",
                               sort=Sort.NEWEST, count=200,
                               continuation_token=token)
        rows += batch
        if not batch or token is None:
            break
        time.sleep(1)
    if not rows:
        raise ValueError("no reviews returned")
    df = pd.DataFrame(rows)[["reviewId", "score", "at", "content"]]
    df.columns = ["review_id", "rating", "date", "review_text"]
    df["app_name"] = app_name
    df["domain"] = info["domain"]
    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    df["review_type"] = df["rating"].apply(review_type)
    return df[["review_id","app_name","domain","rating","date","review_text","review_type"]]

In [4]:
for name, info in APPS.items():
    try:
        d = app(info["id"], lang="en", country="eg")
        print(f"{name:15} -> {d['title'][:30]:30} | {d['developer'][:20]:20} | {d['ratings']:,}")
    except Exception as e:
        print(f"{name:15} -> ERROR: {e}")

Vodafone Egypt  -> Ana Vodafone                   | Vodafone Egypt͏      | 2,340,038
Orange Egypt    -> Orange Max it - Egypt          | Orange Egypt         | 1,016,457
WE              -> My WE                          | Telecom Egypt        | 557,928
Etisalat Cash   -> e& money - EG                  | e& Egypt             | 52,030
Instapay        -> InstaPay Egypt                 | Egyptian Banks Compa | 112,746
Fawry           -> myfawry                        | Fawry for Banking &  | 178,203
NBE             -> NBE Mobile                     | National Bank of Egy | 30,296
CIB             -> CIB Mobile Banking App         | CIB Egypt            | 11,298
Banque Misr     -> BM Online                      | Banque Misr          | 277,217
Amazon          -> Amazon Shopping                | Amazon Mobile LLC    | 4,707,534
Jumia           -> JUMIA Online Shopping          | JUMIA                | 2,724,726
Noon            -> noon Shopping, Food, Grocery   | noon e-commerce      | 1,159,6

In [5]:
OUT = "../../data/raw/by_app"
os.makedirs(OUT, exist_ok=True)

for name, info in APPS.items():
    path = f"{OUT}/{name.replace(' ', '_')}.csv"
    if os.path.exists(path):
        print(f"skip   {name} (already saved)")
        continue
    try:
        df = fetch_app(name, info, target=4000).head(4000)
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"saved  {name:15} {len(df):5} rows")
    except Exception as e:
        print(f"FAILED {name:15} {e}")

saved  Vodafone Egypt   4000 rows
saved  Orange Egypt     4000 rows
saved  WE               4000 rows
saved  Etisalat Cash    4000 rows
saved  Instapay         4000 rows
saved  Fawry            4000 rows
saved  NBE              4000 rows
saved  CIB              3209 rows
saved  Banque Misr      4000 rows
saved  Amazon           4000 rows
saved  Jumia            4000 rows
saved  Noon             4000 rows
saved  Uber             4000 rows
saved  inDrive          4000 rows
saved  Careem           4000 rows
saved  DiDi             4000 rows


In [6]:
files = [f"{OUT}/{f}" for f in os.listdir(OUT) if f.endswith(".csv")]
raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
raw = raw.drop_duplicates(subset="review_id")

raw.to_csv("../../data/raw/raw_data.csv", index=False, encoding="utf-8-sig")
print(raw.groupby(["domain", "app_name"]).size())
print(raw["review_type"].value_counts(normalize=True).round(2))

domain              app_name      
Banking             Banque Misr       4000
                    CIB               3209
                    NBE               4000
E-commerce          Amazon            4000
                    Jumia             4000
                    Noon              4000
Payments            Etisalat Cash     4000
                    Fawry             4000
                    Instapay          4000
Telecom             Orange Egypt      4000
                    Vodafone Egypt    4000
                    WE                4000
Transport Services  Careem            4000
                    DiDi              4000
                    Uber              4000
                    inDrive           4000
dtype: int64
review_type
Positive    0.62
Negative    0.35
Neutral     0.03
Name: proportion, dtype: float64


In [ ]:
raw = pd.read_csv("../../data/raw/raw_data.csv")
raw["review_text"] = raw["review_text"].fillna("")

print("== Time Period ==")
print(raw.groupby("app_name")["date"].agg(["min", "max"]))

print("\n== Star Rating Distribution % ==")
print((pd.crosstab(raw["app_name"], raw["review_type"], normalize="index") * 100).round(0))

print("\n== Text Quality ==")
print("Empty texts:", (raw["review_text"].str.strip() == "").sum())
has_ar = raw["review_text"].str.contains(r"[\u0600-\u06FF]")
print("The percentage containing Arabic:")
print(has_ar.groupby(raw["app_name"]).mean().round(2))

== Time Period ==
                       min         max
app_name                              
Amazon          2025-12-03  2026-09-29
Banque Misr     2026-01-10  2026-09-29
CIB             2024-11-06  2026-09-29
Careem          2025-06-05  2026-09-29
DiDi            2026-03-22  2026-09-29
Etisalat Cash   2025-10-31  2026-09-29
Fawry           2025-11-08  2026-09-29
Instapay        2026-03-07  2026-09-29
Jumia           2023-07-28  2026-09-28
NBE             2024-04-24  2026-09-29
Noon            2026-01-18  2026-09-29
Orange Egypt    2026-08-31  2026-09-29
Uber            2026-07-13  2026-09-29
Vodafone Egypt  2026-07-29  2026-09-29
WE              2026-08-24  2026-09-29
inDrive         2026-07-15  2026-09-29

== Star Rating Distribution % ==
review_type     Negative  Neutral  Positive
app_name                                   
Amazon              43.0      3.0      54.0
Banque Misr         30.0      3.0      67.0
CIB                 55.0      3.0      42.0
Careem              42.0  

In [11]:
raw = pd.read_csv("../../data/raw/raw_data.csv")
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)


raw.sample(50, random_state=1)

,review_id,app_name,domain,rating,date,review_text,review_type
25570,ccb9bc27-49be-4c41-a917-d154122acca7,Fawry,Payments,1,2026-03-31,بي يقول ان الرقم غير صالح رغم اني بكتب الرقم بشكل صحيح,Negative
31987,69a798ef-28bc-438b-b542-76a8eee1f185,Instapay,Payments,1,2026-08-12,طيب، وما الحل الآن؟ البرنامج لا يعمل بعد التحديث,Negative
30467,bafde333-f847-4909-a723-dc90c3e268fa,inDrive,Transport Services,2,2026-07-30,سئ وكيزيدو بزاف,Negative
30157,650dfd01-9a70-45b5-8701-5efa2737aa9a,inDrive,Transport Services,5,2026-08-06,خدمة في قمة الروعة وفقكم الله,Positive
47171,b8c2604e-5490-490a-b722-337d71375004,Noon,E-commerce,5,2026-01-19,جيد جدا,Positive
54985,c016ce07-8dd4-49dd-8117-7069aa3365f1,Uber,Transport Services,1,2026-07-18,"Dear Uber Support, I would like to share my concern regarding the trips I receive through the app. Most of the ride requests I get are very far from my current location, and the pickup payment is ...",Negative
18972,386bb7ba-94ef-45f4-beb7-d81508d706fc,DiDi,Transport Services,5,2026-04-01,رائع,Positive
8639,733d9ac9-59d0-4e5a-8444-cf5ce06a5541,Careem,Transport Services,1,2026-07-03,لدي مشكله عدم الحجز تواصلت مع الدعم ولم يتم حل المشكله,Negative
12376,fccae8a8-dddc-4c14-a129-f82c684cf5f5,CIB,Banking,1,2026-06-05,تم عمل مسح للبيانات ومازالت رساله حدث خطأ مستمره مش عارف ادخل علي البرنامج ما الحل ارجوا الافتده,Negative
58536,10352884-d9ae-4fa0-a0fe-5666f0a05c1d,Vodafone Egypt,Telecom,5,2026-08-10,تجربه جميله جدا,Positive
